# Part 3: Mechanisms, Heterogeneity & Advanced Extensions

Having established a baseline effect, this notebook dives deeper into the *mechanisms* driving the Catholic fertility response. Was it an urban vs. rural phenomenon? Was it driven by Polish nationalism or purely religious friction?


### 1. Setup and Data Loading


In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Enable autoreload for development
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Paths
DATA_RAW = project_root / "data" / "raw"
DATA_PROCESSED = project_root / "data" / "processed"
OUTPUTS = project_root / "outputs" / "figures"
OUTPUTS.mkdir(exist_ok=True, parents=True)

print("Setup complete. Outputs will be saved to:", OUTPUTS)

from src.analysis.exploratory import (
    heterogeneity_by_urbanization, polish_vs_german_catholics, fertility_convergence,
    marriage_to_birth_pipeline, dose_response_plot, infant_mortality_did,
)
from src.analysis.advanced import (
    rollback_event_study, illegitimacy_analysis, infant_mortality_analysis,
    franco_prussian_war_analysis, robustness_exclude_war, trend_adjusted_did,
    polish_german_rollback, placebo_test,
)
from src.data.merge_ipehd import merge_ipehd_controls

panel = pd.read_parquet(DATA_PROCESSED / "analysis_panel.parquet")

# Merge iPEHD controls (lnpop, f_urban, f_jew, f_young, hhsize, ...) so the
# matched-sample / balance-table routines have pre-treatment characteristics
# to work with.
panel = merge_ipehd_controls(panel)


### 2. Heterogeneity Analysis
The Kulturkampf may have impacted urban Catholics (who faced modern state bureaucracy daily) differently from rural Catholics (where traditional parish structures remained resilient). We also separate the effect for Polish vs. German Catholics to untangle religious oppression from ethnic suppression.


In [ ]:
print("=" * 60)
print("HETEROGENEITY: URBAN vs RURAL")
print("=" * 60)
het_results = heterogeneity_by_urbanization(panel, outcome="cbr")

print("\n" + "=" * 60)
print("POLISH vs GERMAN CATHOLICS")
print("=" * 60)
pol_results = polish_vs_german_catholics(panel, outcome="cbr")

### 3. Fertility Convergence and Dose Response
Was the Catholic fertility bump merely a delay in their demographic transition? We examine long-term convergence and estimate a non-linear dose-response curve based on varying intensities of Catholic populations.


In [ ]:
fig_conv, conv_data = fertility_convergence(panel)
fig_conv.savefig(OUTPUTS / "fig7_convergence.png", dpi=300, bbox_inches="tight")
plt.show()

fig_dose, dose_data = dose_response_plot(panel, savepath=str(OUTPUTS / "fig8_dose_response.png"))
plt.show()

### 4. Mechanisms: Marriage Pipeline and Infant Mortality
Demographically, a rise in the crude birth rate could result from earlier/more frequent marriages (the nuptiality channel) or a decline in infant mortality (survivor bias in registration). We test these specific channels.


In [ ]:
lag_results = marriage_to_birth_pipeline(panel)
imr_result = infant_mortality_did(panel)

ill = illegitimacy_analysis(panel)
ill["fig"].savefig(OUTPUTS / "fig10_illegitimacy.png", dpi=300, bbox_inches="tight")
plt.show()

imr = infant_mortality_analysis(panel)
imr["fig"].savefig(OUTPUTS / "fig11_infant_mortality.png", dpi=300, bbox_inches="tight")
plt.show()



### 5. Historical Confounders: The Franco-Prussian War & The Rollback
The Franco-Prussian war (1870-1871) caused severe demographic shocks precisely when our pre-trend window closes. Furthermore, Bismarck began 'rolling back' Kulturkampf laws in the late 1870s. We explicitly model both of these historical shocks to clean our estimates.


In [ ]:
war = franco_prussian_war_analysis(panel)
war["fig"].savefig(OUTPUTS / "fig12_war_analysis.png", dpi=300, bbox_inches="tight")
plt.show()

# Robustness excluding the war years
war_rob = robustness_exclude_war(panel, outcome="cbr", war_years=(1870, 1871, 1872), ref_year=1869, savepath=str(OUTPUTS / "fig13_war_excluded_event_study.png"))
plt.show()

# Examining the Rollback period
rollback = rollback_event_study(panel, outcome="cbr", treatment_var="cath_share", ref_year=1872, savepath=str(OUTPUTS / "fig9_rollback_event_study.png"))
plt.show()

### 6. Final Robustness: Trend Adjustments & Placebo Tests
To conclude our empirical verification, we adjust for differential pre-trends explicitly and run placebo tests on arbitrary years to ensure our model isn't just picking up systemic noise.


In [ ]:
ta_main = trend_adjusted_did(panel, outcome="cbr", trend_base_year=1862, exclude_war=False)
plc = placebo_test(panel, outcome="cbr", placebo_years=[1864, 1866, 1868, 1870, 1873, 1876, 1880, 1884], savepath=str(OUTPUTS / "fig15_placebo_test.png"))
plt.show()



In [ ]:
# %% Cell 37: Pre-treatment balance table (Option C)
from src.analysis.matching_robustness import pretreatment_balance_table

balance = pretreatment_balance_table(panel)

# Save for the paper
balance.to_csv(OUTPUTS / "table1_balance.csv", index=False)

In [ ]:
# %% Cell 38: Matched-sample DiD (Option B)
from src.analysis.matching_robustness import matched_sample_did

matched_results = matched_sample_did(
    panel,
    matching_vars=["lnpop", "f_urban", "f_jew", "f_young", "hhsize"],
    outcome="cbr",
    n_bins=4,
)